# **Capstone Project: A Bi-Objective Evolutionary Approach to Feature Selection for Customer Value Prediction in Fintech**

# *Advanced Modelling & Optimization*

## MBAI 5600G: Applied Integrative Analytics Capstone Project

### Group 7: Brennan Mason & Mohammad Shah

## Environment Setup

In [ ]:
# Specify base path to local directory
BASE_PATH = "/content/drive/Shareddrives/MBAI Capstone S S26 Group 7/"

In [ ]:
!pip install great_tables
!pip install pymoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.2/607.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.0/451.0 kB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.3/328.3 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.9/866.9 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


## Data Ingestion

In [ ]:
import numpy as np
import pandas as pd

# Configure global display format
pd.options.display.float_format = '{:,.5f}'.format

# Define path
path = BASE_PATH + "p2p-customer-value-prediction/data/processed/cleaned_loans.csv"

# Load data
loans = pd.read_csv(path)

# Inspect head
loans.head()

,funded_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,...,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,debt_settlement_flag,rar,issue_yr,delinq_recency_bin
0,"7,000.00000",36,0.07890,A,A5,10.00000,MORTGAGE,"69,000.00000",Not Verified,Fully Paid,...,0.00000,"231,893.00000","29,326.00000","9,100.00000","24,693.00000",Cash,N,"7,126.31286",2015,Never Delinquent
1,"5,000.00000",36,0.06030,A,A1,1.00000,RENT,"37,000.00000",Verified,Fully Paid,...,0.00000,"11,423.00000","1,411.00000","8,200.00000","2,625.00000",Cash,N,"4,944.31630",2012,Never Delinquent
2,"10,000.00000",36,0.07120,A,A3,3.00000,RENT,"100,000.00000",Source Verified,Fully Paid,...,0.00000,"196,745.00000","143,072.00000","27,500.00000","161,126.00000",Cash,N,"9,830.12403",2014,Never Delinquent
3,"12,000.00000",36,0.05930,A,A1,NaN,RENT,"60,000.00000",Verified,Fully Paid,...,0.00000,"206,735.00000","12,162.00000","76,700.00000",0.00000,Cash,N,"11,681.85155",2015,Never Delinquent
4,"6,700.00000",36,0.05320,A,A1,3.00000,MORTGAGE,"69,000.00000",Not Verified,Fully Paid,...,0.00000,"204,505.00000","27,059.00000","52,600.00000","35,900.00000",Cash,N,"6,753.79570",2017,Never Delinquent


In [ ]:
# Specify leaky features to drop
leaky_cols = [
    "total_pymnt",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "loan_status",
    "last_pymnt_amnt",  # Fixing lingering leakage
    "last_fico_range_low",
    "last_fico_range_high",
    "debt_settlement_flag"
]

# Drop leaky features
loans.drop(leaky_cols, axis=1, inplace=True)

## Data Preprocessing

In [ ]:
# Import required classes and functions for preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline

In [ ]:
# Get lists of skewed and symmetric numeric features
num_cols = loans.select_dtypes(include=[np.number]).columns.tolist()
target_col = "rar"
num_feats = [col for col in num_cols if col != target_col]

skew_num_feats = [col for col in num_feats if abs(loans[col].skew()) > 1]  # Slight data leakage occurring here
sym_num_feats = [col for col in num_feats if col not in skew_num_feats]

In [ ]:
# Construct partial preprocessing pipeline for categorical features
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),  # Using mode imputation strategy for categorical features
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Construct full preprocessing pipeline for experimental condition 1 (i.e., without transformation)
raw_preprocessing_pipe = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, make_column_selector(dtype_include=object)),
        ("sym_num", SimpleImputer(strategy="mean"), sym_num_feats),  # Using mean imputation strategy for symmetric numeric features
        ("skew_num", SimpleImputer(strategy="median"), skew_num_feats)  # Using median imputation strategy for skewed numeric features (more robust)
    ],
    verbose_feature_names_out=False
)

# Construct simpler, more flexible/dynamic preprocessing pipeline for GA loops (no static column selection)
flex_raw_preprocessing_pipe = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, make_column_selector(dtype_include=object)),
        ("num", SimpleImputer(strategy="median"), make_column_selector(dtype_include=np.number))
    ],
    verbose_feature_names_out=False
)

# Construct partial preprocessing pipeline for symmetric numeric features with transformation
sym_num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="mean")),
    ("standardize", StandardScaler())
])

# Construct partial preprocessing pipeline for skewed numeric features with transformation
skew_num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("transform", PowerTransformer())  # Performs Yeo-Johnson transformation and then standard scaling
])

# Construct full preprocessing pipeline for subsequent experimental conditions (i.e., with transformation)
trans_preprocessing_pipe = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, make_column_selector(dtype_include=object)),
        ("sym_num", sym_num_pipe, sym_num_feats),
        ("skew_num", skew_num_pipe, skew_num_feats)
    ],
    verbose_feature_names_out=False
)

## Data Partitioning

In [ ]:
# Separate feature matrix and target vector
X = loans.drop(target_col, axis=1)
y = loans[target_col].copy()

## Advanced Modelling & Optimization

In [ ]:
import pickle

# Load pre-populated eval dict
with open(BASE_PATH + "p2p-customer-value-prediction/outputs/eval_dict.pkl", "rb") as f:
  eval_dict = pickle.load(f)

In [ ]:
from sklearn.model_selection import RepeatedKFold

# Define 5x2 CV procedure
cv = RepeatedKFold(n_splits=2, n_repeats=5, random_state=42)

# Specify eval metrics
scoring = ["r2", "neg_mean_absolute_error", "neg_mean_absolute_percentage_error"]

# Define function to compute adjusted R-squared
def compute_adj_r2(r2, n_obs, n_feats):
  """
  Computes the Adjusted R-squared metric.

  Parameters:
  -----------
      - r2 (float): R-squared metric.
      - n_obs (int): Number of observations.
      - n_feats (int): Number of features.

  Returns:
  --------
      - float: Adjusted R-squared metric.
  """
  # Prevent div by zero
  if n_obs <= n_feats + 1:
    return np.nan

  return 1 - (1 - r2) * (n_obs - 1) / (n_obs - n_feats - 1)

# Define function to get clean evaluation summary
def get_eval_summary(scores, X, cond_num, model_name):
  """
  Extracts, calculates, stores, and displays aggregate evaluation metrics from cross-validation.

  Parameters:
  -----------
      - scores (dict): Dictionary of cross-validation scores.
      - X (pd.DataFrame): Full feature matrix used in the experiment.
      - cond_num (int): Experimental condition number.
      - model_name (str): Name of the model/hybrid framework being evaluated.

  Returns:
  --------
      - None
  """
  # Get number of observations in validation set
  n_obs = len(X) / 2

  # Get number of features used by final regressor in pipeline
  if "estimator" in scores:
    try:
      # Extract number of features used during each fold and take average
      fold_feats = [est[-1].n_features_in_ for est in scores["estimator"]]
      n_feats = np.mean(fold_feats)
    except (AttributeError, IndexError):
      n_feats = X.shape[1]
  else:
    n_feats = X.shape[1]

  # Compute aggregate eval metrics
  r2 = round(scores["test_r2"].mean(), 5)
  adj_r2 = round(compute_adj_r2(r2, n_obs, n_feats), 5)
  mae = round(-scores["test_neg_mean_absolute_error"].mean(), 5)
  mape = round(-scores["test_neg_mean_absolute_percentage_error"].mean(), 5)
  fit_time = round(scores["fit_time"].mean(), 5)
  score_time = round(scores["score_time"].mean(), 5)

  # Store results in global eval dict
  eval_dict["Condition"].append(cond_num)
  eval_dict["Model"].append(model_name)
  eval_dict["R-squared"].append(r2)
  eval_dict["Adjusted R-squared"].append(adj_r2)
  eval_dict["MAE"].append(mae)
  eval_dict["MAPE"].append(mape)
  eval_dict["Fit Time (s)"].append(fit_time)
  eval_dict["Score Time (s)"].append(score_time)
  eval_dict["Feature Count"].append(n_feats)

  # Display results
  eval_df = pd.DataFrame({
      key: [val[-1]] for key, val in eval_dict.items() if key not in ["Condition", "Model"]
  }).T.reset_index().rename(columns={"index": "Metric", 0: "Value"})

  print(f"=== Results: Condition {cond_num} | {model_name} ===")
  display(eval_df)

### Experimental Condition 4

In [ ]:
import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

# EC4 pipeline
EC4_GB = Pipeline([
    ("preprocessing", trans_preprocessing_pipe),
    ("pca", PCA(n_components=0.95)),  # Retaining 95% of info
    ("model", xgb.XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        subsample=0.7,
        random_state=42 ))
])

# 5x2 CV
EC4_scores = cross_validate(
    EC4_GB,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

# Results
get_eval_summary(EC4_scores, X, 4, "PCA + GB")

=== Results: Condition 4 | PCA + GB ===


,Metric,Value
0,R-squared,0.71826
1,Adjusted R-squared,0.71816
2,MAE,"3,250.45241"
3,MAPE,0.49619
4,Fit Time (s),29.26074
5,Score Time (s),2.45179
6,Feature Count,45.00000


### Experimental Condition 5

In [ ]:
from pymoo.core.problem import Problem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.repair.rounding import RoundingRepair
from pymoo.operators.sampling.rnd import IntegerRandomSampling
from pymoo.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

feature_cols = X.columns.tolist()
n_features = len(feature_cols)  # 65 raw features

# Single 80/20 split for GA fitness evaluation
X_train_ga, X_test_ga, y_train_ga, y_test_ga = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Single-Objective Feature Selection Problem
class FeatureSelectionProblem(Problem):
    def __init__(self):
        super().__init__(
            n_var=n_features,
            n_obj=1,
            vtype=int,
            xl=0.0,
            xu=1.0
        )

    def __evaluate_one(self, x) -> float:
        # Penalize empty solutions
        if np.sum(x) == 0:
            return 1e10

        # Select features based on binary mask
        selected_cols = X.columns[x == 1].tolist()

        X_train_sub = X_train_ga[selected_cols]
        X_test_sub = X_test_ga[selected_cols]

        # Raw preprocessing + XGBoost (EC1 style, no scaling)
        pipe = Pipeline([
            ("preprocess", flex_raw_preprocessing_pipe),
            ("model", xgb.XGBRegressor(
                tree_method="hist",
                n_jobs=-1,
                subsample=0.7,
                random_state=42))])

        pipe.fit(X_train_sub, y_train_ga)
        y_pred = pipe.predict(X_test_sub)

        return mean_absolute_error(y_test_ga, y_pred)

    def _evaluate(self, x, out, *args, **kwargs):
        objectives = np.zeros(len(x))

        for i, _x in enumerate(x):
            objectives[i] = self.__evaluate_one(_x)

        out["F"] = objectives

problem = FeatureSelectionProblem()

# GA algorithm
algorithm = GA(
    pop_size=50,  # Reduced for Colab feasibility
    sampling=IntegerRandomSampling(),
    crossover=SBX(prob=1.0, eta=3.0, vtype=float, repair=RoundingRepair()),
    mutation=PM(prob=1.0, eta=3.0, vtype=float, repair=RoundingRepair()),
    eliminate_duplicates=True )

res = minimize(problem,
               algorithm,
               termination = ("n_gen", 20),
               seed=42,
               save_history=True,
               verbose=True)

# Extract Best Solution
best_mae = res.F[0]
print(f"\nBest MAE (GA fitness): {best_mae:.2f}")

selected_cols = X.columns[res.X == 1].tolist()
print(f"Number of features selected: {len(selected_cols)} out of {n_features}")
print(f"Selected features: {selected_cols}")

# 5x2 CV
EC5_GB = Pipeline([
    ("preprocessing", flex_raw_preprocessing_pipe),
    ("model", xgb.XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        subsample=0.7,
        random_state=42))])

EC5_scores = cross_validate(
    EC5_GB,
    X[selected_cols],
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

print()
get_eval_summary(EC5_scores, X[selected_cols], 5, "SGA + GB")

n_gen  |  n_eval  |     f_avg     |     f_min    
     1 |       50 |  3.868785E+03 |  2.603670E+03
     2 |      100 |  2.624050E+03 |  2.603670E+03
     3 |      150 |  2.613533E+03 |  2.598470E+03
     4 |      200 |  2.606004E+03 |  2.598470E+03
     5 |      250 |  2.602238E+03 |  2.596599E+03
     6 |      300 |  2.599841E+03 |  2.593252E+03
     7 |      350 |  2.597061E+03 |  2.589283E+03
     8 |      400 |  2.594451E+03 |  2.586350E+03
     9 |      450 |  2.592247E+03 |  2.582433E+03
    10 |      500 |  2.590305E+03 |  2.581770E+03
    11 |      550 |  2.589053E+03 |  2.581770E+03
    12 |      600 |  2.587984E+03 |  2.581770E+03
    13 |      650 |  2.587291E+03 |  2.581770E+03
    14 |      700 |  2.586679E+03 |  2.581770E+03
    15 |      750 |  2.586114E+03 |  2.580845E+03
    16 |      800 |  2.585758E+03 |  2.580845E+03
    17 |      850 |  2.585354E+03 |  2.580507E+03
    18 |      900 |  2.584844E+03 |  2.580507E+03
    19 |      950 |  2.584471E+03 |  2.580507E+03


,Metric,Value
0,R-squared,0.75856
1,Adjusted R-squared,0.75840
2,MAE,"2,606.39374"
3,MAPE,0.39890
4,Fit Time (s),9.48864
5,Score Time (s),1.06844
6,Feature Count,86.00000


### Experimental Condition 6

In [ ]:
import plotly.graph_objects as go

# Import pymoo components for optimization
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.optimize import minimize
from pymoo.core.callback import Callback
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

# Import sklearn functions for modelling and eval
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

# Perform 80/20 train-test split for GA fitness eval
X_train_ga, X_test_ga, y_train_ga, y_test_ga = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Get initial feature names and count
feature_cols = X.columns.tolist()
n_features = len(feature_cols)

# Define bi-objective feature selection problem
class BiObjectiveFeatureSelectionProblem(Problem):
  def __init__(self):
    super().__init__(
        n_var=n_features,  # One chromosome per feature
        n_obj=2,  # Bi-objective problem (min MAE, feature count)
        n_constr=1,  # Set min feature count
        xl=0,
        xu=1,
        vtype=bool
    )

  def __evaluate_one(self, x):
    # Compute number of features selected
    n_selected = np.sum(x)

    # Handle empty solutions
    if n_selected == 0:
      return 1e10, 0

    # Extract selected features
    mask = x.astype(bool)
    selected_cols = X.columns[mask].tolist()
    X_train_sub = X_train_ga[selected_cols]
    X_test_sub = X_test_ga[selected_cols]

    # Define pipeline config (raw preprocessing + XGBoost)
    pipe = make_pipeline(
        flex_raw_preprocessing_pipe,
        xgb.XGBRegressor(
            tree_method="hist",
            n_jobs=-1,
            subsample=0.7,
            random_state=42
        )
    )

    # Train model
    pipe.fit(X_train_sub, y_train_ga)

    # Make preds on test set
    y_pred = pipe.predict(X_test_sub)

    # Compute MAE
    mae = mean_absolute_error(y_test_ga, y_pred)

    # Return MAE and feature count
    return mae, n_selected

  def _evaluate(self, x, out, *args, **kwargs):
    F = []  # Objectives
    G = []  # Constraints

    for _x in x:
      mae, n_feats = self.__evaluate_one(_x)
      F.append([mae, n_feats])  # Minimize MAE, feature count
      G.append([1 - n_feats])  # Ensure at least 1 feature selected

    out["F"] = np.array(F)
    out["G"] = np.array(G)

# Define callback to display only non-dominated solutions
class NSGA2Callback(Callback):
  def notify(self, algorithm):
    gen = algorithm.n_gen
    F = algorithm.pop.get("F")

    # Get non-dominated solution indices
    nds = NonDominatedSorting()
    front = nds.do(F, only_non_dominated_front=True)

    print(f"\n=== Generation {gen} ===")
    print("\nMAE\t\tFeature Count")
    print("-" * 30)
    for idx in front:
      mae = F[idx][0]
      n_feats = int(F[idx][1])
      print(f"{mae:.4f}\t\t{n_feats}")

problem = BiObjectiveFeatureSelectionProblem()

algorithm = NSGA2(
    pop_size=50,
    sampling=BinaryRandomSampling(),
    crossover=TwoPointCrossover(),
    mutation=BitflipMutation(),
    eliminate_duplicates=True
)

res = minimize(
    problem,
    algorithm,
    termination=("n_gen", 20),
    seed=42,
    save_history=True,
    verbose=False,
    callback=NSGA2Callback()
)


=== Generation 1 ===

MAE		Feature Count
------------------------------
2619.1543		29
2628.5277		27
2611.6929		30
5658.3500		25
2603.0549		34

=== Generation 2 ===

MAE		Feature Count
------------------------------
2611.6929		30
5550.0404		24
2612.9589		27
2635.7844		26
2597.2811		31

=== Generation 3 ===

MAE		Feature Count
------------------------------
5550.0404		24
2612.9589		27
2597.2811		31
2619.0428		26
2603.5217		30
2610.5028		28
2625.9135		25
5958.1666		22
2607.0040		29

=== Generation 4 ===

MAE		Feature Count
------------------------------
2597.2811		31
2595.2928		35
5864.2965		19
2607.3643		24
2597.8858		27
2595.0569		36

=== Generation 5 ===

MAE		Feature Count
------------------------------
5864.2965		19
2597.8858		27
2587.3008		34
2602.7676		22
5561.6171		20
2596.6927		30

=== Generation 6 ===

MAE		Feature Count
------------------------------
2597.8858		27
2587.3008		34
2602.7676		22
5561.6171		20
2601.4970		23
5855.6703		18
2592.8715		30
2593.7209		29
2605.0628		21

=

In [ ]:
# Extract and visualize Pareto optimal solutions
print("\nFINAL PARETO OPTIMAL SOLUTIONS")
print("=" * 60)
for i in range(len(res.F)):
  mae = res.F[i][0]
  n_feats = int(res.F[i][1])
  selected_feats = X.columns[res.X[i].astype(bool)].tolist()
  print(f"\nSolution {i+1}")
  print("-" * 40)
  print(f"MAE: {mae:.4f}")
  print(f"Feature Count: {n_feats}")
  print(f"Selected Features: {selected_feats}")

print()

pareto_mae = [f[0] for f in res.F]
pareto_n_feats = [int(f[1]) for f in res.F]

pareto_df = pd.DataFrame({
    "MAE": pareto_mae,
    "Feature Count": pareto_n_feats
})

pareto_df.sort_values(by="Feature Count", ascending=True, inplace=True)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pareto_df["Feature Count"],
    y=pareto_df["MAE"],
    mode="lines+markers",
    line=dict(color="steelblue", dash="dash"),
    marker=dict(color="steelblue", size=10),
    name="Pareto Optimal Solutions",
    hovertemplate="Features: %{x}<br>MAE: %{y:.4f}<extra></extra>"
))

fig.update_layout(
    title="Pareto Front: MAE vs Feature Count",
    xaxis_title="Feature Count",
    yaxis_title="Mean Absolute Error (MAE)",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600
)

fig.update_yaxes(ticklabelstandoff=5, tickformat=",")
fig.update_xaxes(ticklabelstandoff=5)

fig.show()


FINAL PARETO OPTIMAL SOLUTIONS

Solution 1
----------------------------------------
MAE: 2605.5612
Feature Count: 16
Selected Features: ['funded_amnt', 'term', 'int_rate', 'sub_grade', 'home_ownership', 'annual_inc', 'inq_last_6mths', 'initial_list_status', 'collections_12_mths_ex_med', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'pub_rec_bankruptcies', 'tot_hi_cred_lim', 'issue_yr', 'delinq_recency_bin']

Solution 2
----------------------------------------
MAE: 5724.0252
Feature Count: 11
Selected Features: ['sub_grade', 'annual_inc', 'pub_rec', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_tl_op_past_12m', 'tot_hi_cred_lim', 'total_bc_limit', 'issue_yr', 'delinq_recency_bin']

Solution 3
----------------------------------------
MAE: 2603.5201
Feature Count: 17
Selected Features: ['funded_amnt', 'term', 'int_rate', 'home_ownership', 'annual_inc', 'fico_range_low', 'initial_list_status', 'collections_12_mths_ex_med', 'acc_now_delinq',

In [ ]:
EC6_GB = make_pipeline(
    flex_raw_preprocessing_pipe,
    xgb.XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        subsample=0.7,
        random_state=42
    )
)

# Extract solution 8 feature set (most parsimonious)
selected_cols = ['funded_amnt', 'term', 'sub_grade', 'annual_inc', 'pub_rec', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_op_rev_tl', 'pub_rec_bankruptcies', 'tot_hi_cred_lim', 'issue_yr', 'delinq_recency_bin']

EC6_scores = cross_validate(
    EC6_GB,
    X[selected_cols],
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

get_eval_summary(EC6_scores, X[selected_cols], cond_num=6, model_name="NSGA-II + GB")

=== Results: Condition 6 | NSGA-II + GB ===


,Metric,Value
0,R-squared,0.75633
1,Adjusted R-squared,0.75624
2,MAE,"2,628.86346"
3,MAPE,0.40126
4,Fit Time (s),5.45216
5,Score Time (s),0.88970
6,Feature Count,49.00000


### Experimental Condition 7 (HPO)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint

# Define model pipeline
EC7_GB = make_pipeline(
    flex_raw_preprocessing_pipe,
    xgb.XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        subsample=0.7,
        random_state=42
    )
)

# Define distribution of params to test
param_dist = {
    "xgbregressor__n_estimators": randint(100, 300),
    "xgbregressor__learning_rate": loguniform(0.01, 0.3),
    "xgbregressor__max_depth": randint(3, 10)
}

# Initialize and execute randomized search
rnd_search = RandomizedSearchCV(
    EC7_GB,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1,
    random_state=42
)

rnd_search.fit(X[selected_cols], y)

# Extract results
print(f"\nBest hyperparameters found: {rnd_search.best_params_}")
final_model = rnd_search.best_estimator_

Fitting 10 folds for each of 50 candidates, totalling 500 fits

Best hyperparameters found: {'xgbregressor__learning_rate': np.float64(0.049833191601257244), 'xgbregressor__max_depth': 7, 'xgbregressor__n_estimators': 274}


In [ ]:
EC7_scores = cross_validate(
    final_model,
    X[selected_cols],
    y,
    cv=cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1
)

get_eval_summary(EC7_scores, X[selected_cols], cond_num=7, model_name="HPO + GB")

=== Results: Condition 7 | HPO + GB ===


,Metric,Value
0,R-squared,0.76583
1,Adjusted R-squared,0.76574
2,MAE,"2,606.69845"
3,MAPE,0.39937
4,Fit Time (s),14.06211
5,Score Time (s),1.90398
6,Feature Count,49.00000


### Feature Importance Analysis

In [ ]:
# Extract feature importances from CV folds
fold_importances = []
for est in EC7_scores["estimator"]:
  fold_importances.append(est[-1].feature_importances_)

# Get feature names
feature_names = EC7_scores["estimator"][0][0].get_feature_names_out()

df_importances = pd.DataFrame(fold_importances, columns=feature_names)

mean_importances = df_importances.mean().sort_values(ascending=True)

final_model_importances_df = mean_importances.reset_index()
final_model_importances_df.columns = ["Feature", "Importance"]
final_model_importances_df = final_model_importances_df.tail(10)

# Visualize feature importances
fig = go.Figure()

fig.add_trace(go.Bar(
    y=final_model_importances_df["Feature"],
    x=final_model_importances_df["Importance"],
    orientation="h",
    marker=dict(color="steelblue"),
    text=final_model_importances_df["Importance"],
    texttemplate="%{text:.3f}",
    textposition="outside"
))

fig.update_layout(
    title="Global Feature Importance - Final Optimized Model",
    xaxis_title="Mean Importance Score (Gain)",
    yaxis_title="",
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    width=1000,
    height=600,
    margin=dict(l=200, r=50)
)

fig.update_yaxes(ticklabelstandoff=5)
fig.update_xaxes(tickformat=".2f", ticklabelstandoff=5, range=[0, final_model_importances_df["Importance"].max() * 1.10])

fig.show()

### Evaluation Summary

In [ ]:
from great_tables import GT, loc, md

# Create eval summary df
eval_df = pd.DataFrame(eval_dict)

# Create clean eval summary table
eval_tbl = (
    GT(eval_df)
    .tab_header(
        title="Performance Summary by Condition & Model",
        subtitle="10-Fold Cross-Validation Evaluation Results"
    )
    .tab_spanner(
        label="Experiment Configuration",
        columns=["Condition", "Model"]
    )
    .tab_spanner(
        label="Evaluation Metrics",
        columns=["R-squared", "Adjusted R-squared", "MAE", "MAPE", "Fit Time (s)", "Score Time (s)", "Feature Count"],
    )
    .fmt_number(
        columns=["R-squared", "Adjusted R-squared"],
        decimals=5
    )
    .fmt_number(
        columns=["MAE", "Fit Time (s)", "Score Time (s)"],
        decimals=2,
        use_seps=True
    )
    .fmt_percent(
        columns=["MAPE"]
    )
    .fmt_integer(
        columns=["Feature Count"]
    )
    .tab_footnote(
        footnote="All evaluation metrics are averaged across 10 (5x2) cross-validation folds. Temporal metrics are measured in seconds.",
        locations=loc.spanner_labels(ids=["Evaluation Metrics"])
    )
    .tab_footnote(
        footnote="Feature count indicates the preprocessed inputs provided to the supervised regressor, excluding features strictly utilized for clustering.",
        locations=loc.column_labels(columns=["Feature Count"])
    )
    .tab_footnote(
        footnote=md("A hyperparameter of *k*=3 was utilized for the *k*-means clustering algorithm across all hybrid frameworks."),
        locations=loc.body(
            columns=["Model"],
            rows=lambda x: x["Model"].isin(["KM + DT", "KM + RF", "KM + GB"])
        )
    )
    .cols_align(
        align="left",
        columns=["Condition"]
    )
    .cols_label(
        **{
            "Fit Time (s)": "Fit Time",
            "Score Time (s)": "Score Time"
        }
    )
    .opt_row_striping()
)

eval_tbl.show()

Performance Summary by Condition & Model 
 
 
 10-Fold Cross-Validation Evaluation Results 
 
 
 
 Experiment Configuration 
 
 
 Evaluation Metrics 1 
 
 
 
 Condition 
 Model 
 R-squared 
 Adjusted R-squared 
 MAE 
 MAPE 
 Fit Time 
 Score Time 
 Feature Count 2 
 
 
 
 
 1 
 DT 
 0.65978 
 0.65944 
 2,763.75 
 41.04% 
 10.09 
 0.66 
 126 
 
 
 1 
 RF 
 0.76628 
 0.76605 
 2,602.21 
 39.91% 
 151.59 
 1.35 
 126 
 
 
 1 
 GB 
 0.75844 
 0.75820 
 2,611.06 
 39.92% 
 7.68 
 0.92 
 126 
 
 
 2 
 DT 
 0.66020 
 0.65986 
 2,762.61 
 41.02% 
 16.57 
 0.99 
 126 
 
 
 2 
 RF 
 0.76618 
 0.76595 
 2,604.10 
 39.92% 
 154.89 
 1.50 
 126 
 
 
 2 
 GB 
 0.75799 
 0.75775 
 2,612.98 
 39.92% 
 14.48 
 1.16 
 126 
 
 
 3 
 KM + DT 3 
 0.41724 
 0.41672 
 5,066.66 
 76.60% 
 16.81 
 1.08 
 113 
 
 
 3 
 KM + RF 3 
 0.60578 
 0.60543 
 4,341.81 
 70.87% 
 144.07 
 1.94 
 113 
 
 
 3 
 KM + GB 3 
 0.58976 
 0.58940 
 4,382.82 
 69.15% 
 18.21 
 1.28 
 113 
 
 
 4 
 PCA + GB 
 0.71826 
 0.71816 
 3,250.45 
 49.62% 
 29.26 
 2.45 
 45 
 
 
 5 
 SGA + GB 
 0.75856 
 0.75840 
 2,606.39 
 39.89% 
 9.49 
 1.07 
 86 
 
 
 6 
 NSGA-II + GB 
 0.75633 
 0.75624 
 2,628.86 
 40.13% 
 5.45 
 0.89 
 49 
 
 
 7 
 HPO + GB 
 0.76583 
 0.76574 
 2,606.70 
 39.94% 
 14.06 
 1.90 
 49 
 
 
 1 All evaluation metrics are averaged across 10 (5x2) cross-validation folds. Temporal metrics are measured in seconds. 2 Feature count indicates the preprocessed inputs provided to the supervised regressor, excluding features strictly utilized for clustering. 3 A hyperparameter of k =3 was utilized for the k -means clustering algorithm across all hybrid frameworks.

In [ ]:
from plotly.subplots import make_subplots

key_experiments = [
    "EC1 - GB",
    "EC4 - PCA + GB",
    "EC5 - SGA + GB",
    "EC6 - NSGA-II + GB",
    "EC7 - HPO + GB"
]

eval_df["Experiment"] = "EC" + eval_df["Condition"].astype(str) + " - " + eval_df["Model"]
key_experiments_df = eval_df[eval_df["Experiment"].isin(key_experiments)]

# Get baseline evals for horizontal line plotting
baseline_mask = key_experiments_df["Experiment"] == "EC1 - GB"
baseline_n_feats = key_experiments_df[baseline_mask]["Feature Count"].values[0]
baseline_r2 = key_experiments_df[baseline_mask]["Adjusted R-squared"].values[0]

# Define func to map bar colours for plot
def map_colours(row):
  exp = row["Experiment"]
  if exp == "EC1 - GB":
    return "darkgrey"  # Grey for baseline model
  elif exp == "EC7 - HPO + GB":
    return "darkorange"  # Orange for optimized model
  else:
    return "steelblue"  # Blue for advanced models

# Map colours for plot
key_experiments_df["Colour"] = key_experiments_df.apply(map_colours, axis=1)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05
)

fig.add_trace(go.Bar(
    x=key_experiments_df["Experiment"],
    y=key_experiments_df["Feature Count"],
    marker=dict(color=key_experiments_df["Colour"]),
    text=key_experiments_df["Feature Count"],
    texttemplate="%{text:.0f}",
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(color="white"),
    showlegend=False
), row=1, col=1)

fig.add_hline(
    y=baseline_n_feats,
    line_dash="dot",
    line_color="darkslategrey",
    line_width=1.5,
    layer="above",
    annotation_text="Baseline",
    annotation_position="top left",
    row=1
)

fig.add_trace(go.Bar(
    x=key_experiments_df["Experiment"],
    y=key_experiments_df["Adjusted R-squared"],
    marker=dict(color=key_experiments_df["Colour"]),
    text=key_experiments_df["Adjusted R-squared"],
    texttemplate="%{text:.4f}",
    textposition="inside",
    insidetextanchor="end",
    textfont=dict(color="white"),
    showlegend=False
), row=2, col=1)

fig.add_hline(
    y=baseline_r2,
    line_dash="dot",
    line_color="darkslategrey",
    line_width=1.5,
    layer="above",
    annotation_text="Baseline",
    annotation_position="top left",
    row=2
)

# Add invisible dummy traces for custom legend mapping
legend_map = [
    ("Baseline", "darkgrey"),
    ("Advanced", "steelblue"),
    ("Optimized", "darkorange")
]

for label, colour in legend_map:
  fig.add_trace(go.Bar(
      x=[None],
      y=[None],
      name=label,
      marker=dict(color=colour),
      showlegend=True
  ), row=1, col=1)

fig.update_layout(
    title=dict(
        text="Model Performance Comparison",
        x=0.5,
        xanchor="center"
    ),
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    legend = dict(
        orientation="h",
        yanchor="bottom",
        y=1.025,
        xanchor="center",
        x=0.5
    ),
    margin=dict(t=120),
    width=1000,
    height=800
)

fig.update_xaxes(title_text="Model", ticklabelstandoff=5, row=2, col=1)
fig.update_yaxes(title_text="Number of Features", tickformat=".0f", ticklabelstandoff=5, rangemode="tozero", row=1, col=1)
fig.update_yaxes(title_text="Adjusted R-squared", tickformat=".1f", ticklabelstandoff=5, rangemode="tozero", row=2, col=1)

fig.show()

/tmp/ipykernel_18585/3893973911.py:30: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=key_experiments_df["Feature Count"],
    y=key_experiments_df["Adjusted R-squared"],
    mode="markers+text",
    textposition=[
        "top center",
        "top center",
        "top center",
        "bottom center",
        "top center"
    ],
    text=key_experiments_df["Experiment"],
    marker=dict(
        size=16,
        color=key_experiments_df["Colour"],
        opacity=0.9
    ),
    showlegend=False
))

fig.add_vline(
    x=baseline_n_feats,
    line_dash="dot",
    line_color="darkslategrey",
    line_width=1.5,
    opacity=0.5
)

fig.add_hline(
    y=baseline_r2,
    line_dash="dot",
    line_color="darkslategrey",
    line_width=1.5,
    opacity=0.5
)

legend_map = [
    ("Baseline", "darkgrey"),
    ("Advanced", "steelblue"),
    ("Optimized", "darkorange")
]

for label, colour in legend_map:
  fig.add_trace(go.Scatter(
      x=[None],
      y=[None],
      mode="markers",
      marker=dict(size=16, color=colour),
      name=label,
      showlegend=True
  ))

fig.update_layout(
    title=dict(
        text="Pareto Front: Complexity vs Performance",
        x=0.5,
        xanchor="center"
    ),
    template="seaborn",
    font=dict(
        family="Helvetica Neue, sans-serif",
        color="#666666"
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.025,
        xanchor="center",
        x=0.5
    ),
    width=1000,
    height=600
)

fig.for_each_trace(lambda t: t.update(textfont_color=t.marker.color))

fig.update_xaxes(
    title_text="Number of Features",
    tickformat=".0f",
    ticklabelstandoff=5,
    rangemode="tozero"
)

fig.update_yaxes(
    title_text="Adjusted R-squared",
    tickformat=".2f",
    ticklabelstandoff=5,
)

fig.show()